# 01 — Data contract

Aligned with `data.md`. Workbooks may be absent (gitignored). The fitted
manifest always records the label audit.


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent):
    if (_candidate / "_paths.py").is_file():
        _nb = str(_candidate.resolve())
        if _nb not in sys.path:
            sys.path.insert(0, _nb)
        break
else:
    raise RuntimeError("Open this notebook from the ValveGuard repo (root or notebooks/).")

import json

from _paths import ARTIFACT_DIR, LABS_WORKBOOK, NOTES_WORKBOOK
from model.tier2_dynamic_competing_risk import (
    DEFAULT_LONGITUDINAL_FEATURE_NAMES,
    DEFAULT_STATIC_FEATURE_NAMES,
)

manifest = json.loads((ARTIFACT_DIR / "manifest.json").read_text(encoding="utf-8"))
print("run_id", manifest["run_id"])
print("eligible_patients", manifest["dataset"]["eligible_patients"])
print("excluded", manifest["dataset"]["excluded"])
audit = manifest["dataset"]["label_audit"]
print("label_status", audit["label_status"])
print("stage_3 templates on BVF clock", audit["stage_3_hemodynamic_templates_on_bvf_clock"])
print("NSVD templates", audit["nsvd_or_non_structural_templates"])
print("IE eligibility yes", audit["eligibility_screen_active_infective_endocarditis_yes"])
print("static width", len(DEFAULT_STATIC_FEATURE_NAMES))
print(DEFAULT_STATIC_FEATURE_NAMES)
print("visit width", len(DEFAULT_LONGITUDINAL_FEATURE_NAMES))
print(DEFAULT_LONGITUDINAL_FEATURE_NAMES)
assert "sts_prom" not in DEFAULT_STATIC_FEATURE_NAMES
assert "known_nsvd" not in DEFAULT_LONGITUDINAL_FEATURE_NAMES
assert "bsa_m2" in DEFAULT_STATIC_FEATURE_NAMES
assert "prosthetic_malposition" in DEFAULT_LONGITUDINAL_FEATURE_NAMES


In [ ]:
from backend.data_pipeline import inspect_expanded_labs, parse_expanded_synthetic_notes

print("notes workbook", NOTES_WORKBOOK.is_file())
print("labs workbook", LABS_WORKBOOK.is_file())
if LABS_WORKBOOK.is_file():
    report = inspect_expanded_labs(LABS_WORKBOOK)
    print("labs joined?", report["joined_into_training_tensor"])
    print("date grain", report["result_date_grain"])
    print(report["reason"])
if NOTES_WORKBOOK.is_file():
    cohort = parse_expanded_synthetic_notes(NOTES_WORKBOOK)
    print("parsed eligible", len(cohort.patients))
    print(cohort.label_audit["structural_reason_nsvd_cause_is_zero"][:280])
else:
    print("Workbooks not in git. Manifest audit above is the shipped evidence.")


12 months is **study inclusion**, not a landmark gate.
The 126 Stage-3 templates are **not CEC BVF**.
Omitted mechanism flags stay `null`.
